# Planetary Motion — Q4
## Part 2 and Part 3 

I've completed the parts of Q4 that don't depend on the orbit simulation from Q2. For Part 2, I've calculated the theoretical orbital parameters for different initial speeds, including the perihelion, aphelion, semimajor axis, semiminor axis and eccentricity. For Part 3, I've used orbital energy to classify the orbits as elliptical, parabolic or hyperbolic. I've also added a few simple tests to check that the classification works. There are still some things left to do. We need the velocity Verlet simulation code from Q2 before we can complete Part 1, which involves changing the initial speed and plotting the different orbits.

Assumptions: I assume the Sun stays at the origin. The planet starts at $\mathbf r_0=R\hat{\mathbf x}$ and moves in the $y$-direction with velocity $\mathbf v_0=v_0\hat{\mathbf y}$, where $v_0>0$. I use SI units unless stated otherwise.


### Constants and energy

We use $G$ for the gravitational constant, $M$ for the mass of the Sun, and $R$ for the starting distance from the Sun. Let $\mu=GM$.

The initial energy per kilogram of the planet is

$$\mathcal E_0=\frac12v_0^2-\frac{GM}{R}.$$

For a circular orbit, the speed is $v_c=\sqrt{GM/R}$.


In [1]:
import numpy as np

# Constants from the project booklet
G = 6.67348e-11
M = 1.989e30
R = 149.6e9

# Energy per unit mass
def energy(v):
    return 0.5 * v**2 - G * M / R

v_circular = np.sqrt(G * M / R)
v_escape = np.sqrt(2 * G * M / R)

print("Circular speed (km/s):", round(v_circular / 1000, 4))


Circular speed (km/s): 29.7871


## Part 2 — Perihelion, aphelion and ellipse parameters

From conservation of angular momentum, $h=Rv_0$. At the closest or furthest point of the orbit, the planet is not moving towards or away from the Sun. So

$$\mathcal E_0=\frac{h^2}{2r^2}-\frac{GM}{r}.$$

The planet starts with tangential velocity, so $R$ is one of these points. Solving the equation gives the other distance:

$$\boxed{r_2=\frac{R^2v_0^2}{2GM-Rv_0^2}}.$$

For an elliptical orbit ($0<v_0<v_{\mathrm{esc}}$), the smaller distance is the perihelion $P$, and the larger one is the aphelion $A$:

$$\boxed{P=\min(R,r_2),\qquad A=\max(R,r_2)}.$$

We then use the formulas from the project booklet to find the semimajor axis $a$, semiminor axis $b$, and eccentricity $\epsilon$:

$$\boxed{a=\frac{A+P}{2},\qquad b=\sqrt{AP},\qquad \epsilon=\frac{A-P}{A+P}}.$$

The example speeds below all give elliptical orbits. The distances are divided by $R$ (about 1 AU).


In [2]:
# Calculate ellipse parameters
def ellipse_values(v):
    r2 = R**2 * v**2 / (2 * G * M - R * v**2)
    P = min(R, r2)
    A = max(R, r2)
    a = (A + P) / 2
    b = np.sqrt(A * P)
    e = (A - P) / (A + P)
    return P, A, a, b, e

# Try different initial speeds (km/s)
speeds = [20, 25, 29.8, 35, 40]
print("Speed | P/R | A/R | a/R | b/R | Eccentricity")
for speed in speeds:
    P, A, a, b, e = ellipse_values(speed * 1000)
    print(speed, round(P / R, 4), round(A / R, 4),
          round(a / R, 4), round(b / R, 4), round(e, 4))

Speed | P/R | A/R | a/R | b/R | Eccentricity
20 0.291 1.0 0.6455 0.5395 0.5492
25 0.5437 1.0 0.7718 0.7374 0.2956
29.8 1.0 1.0017 1.0009 1.0009 0.0009
35 1.0 2.2291 1.6146 1.493 0.3806
40 1.0 9.167 5.0835 3.0277 0.8033


## Part 3 — Orbit classification

We can tell the orbit type from the sign of the specific orbital energy. For a nonzero tangential starting speed:

$$\boxed{\mathcal E_0<0:\ \text{ellipse},\quad
\mathcal E_0=0:\ \text{parabola},\quad
\mathcal E_0>0:\ \text{hyperbola}.}$$

The code allows a small difference from zero, because Python may not calculate decimal numbers exactly.


In [3]:
# Classify the orbit
def orbit_type(v):
    E = energy(v)
    if abs(E) < 0.001:
        return "Parabola"
    if E < 0:
        return "Ellipse"
    return "Hyperbola"

# Compare different initial speeds
speeds = [20, 29.8, 40, v_escape / 1000, 45]
print("Speed (km/s) | Energy (J/kg) | Orbit")
for speed in speeds:
    v = speed * 1000
    print(round(speed, 4), round(energy(v), 2), orbit_type(v))

Speed (km/s) | Energy (J/kg) | Orbit
20 -687269500.0 Ellipse
29.8 -443249500.0 Ellipse
40 -87269500.0 Ellipse
42.1253 0.0 Parabola
45 125230500.0 Hyperbola


In [4]:
# Test the orbit types
print(orbit_type(20000))
print(orbit_type(v_escape))
print(orbit_type(45000))

Ellipse
Parabola
Hyperbola


## Conclusion

- Part 2: We use the starting speed to calculate the theoretical perihelion $P$, aphelion $A$, semimajor axis $a$, semiminor axis $b$, and eccentricity $\epsilon$.
- Part 3: We use the sign of the specific orbital energy to classify the orbit as elliptical, parabolic, or hyperbolic.

Still to do: We need the Q2 orbit simulation to plot trajectories and compare the simulated results with these theoretical values. This notebook does not simulate the orbits yet.
